**Problem 3** b

In [32]:
!pip install -q "lm_eval[hf]" accelerate datasets
!pip install -q langdetect
import torch



models = {
    "Base": "allenai/OLMo-2-0425-1B",
    "SFT": "allenai/OLMo-2-0425-1B-SFT",
    "DPO": "allenai/OLMo-2-0425-1B-DPO",
    "Instruct": "allenai/OLMo-2-0425-1B-Instruct"
}

tasks = ["gsm8k", "ifeval"]

In [33]:
import subprocess

for name, model in models.items():
    for task in tasks:
        print("running", name, "on", task)

        command = [
            "lm-eval", "run",
            "--model", "hf",
            "--model_args", f"pretrained={model},dtype=float16",
            "--tasks", task,
            "--num_fewshot", "0",
            "--limit", "200",
            "--batch_size", "auto",
            "--device", "cuda:0",
            "--gen_kwargs", "do_sample=False",
            "--log_samples",
            "--output_path", f"results/{name}/{task}"
        ]

        if name != "Base":
            command.append("--apply_chat_template")

        subprocess.run(command, check=True)

running Base on gsm8k
running Base on ifeval
running SFT on gsm8k
running SFT on ifeval
running DPO on gsm8k
running DPO on ifeval
running Instruct on gsm8k
running Instruct on ifeval


In [34]:
import json
import glob

for name in models:
    for task in tasks:
        files = sorted(glob.glob(f"results/{name}/{task}/**/results_*.json", recursive=True))

        if files:
            with open(files[-1]) as f:
                results = json.load(f)

            print(name, task)
            print(results["results"])
            print()
        else:
            print("none for", name, task)


Base gsm8k
{'gsm8k': {'name': 'gsm8k', 'alias': 'gsm8k', 'sample_len': 200, 'exact_match,strict-match': 0.0, 'exact_match_stderr,strict-match': 0.0, 'exact_match,flexible-extract': 0.04, 'exact_match_stderr,flexible-extract': 0.013891177924157566}}

Base ifeval
{'ifeval': {'name': 'ifeval', 'alias': 'ifeval', 'sample_len': 200, 'prompt_level_strict_acc,none': 0.095, 'prompt_level_strict_acc_stderr,none': 0.020785455873744915, 'inst_level_strict_acc,none': 0.20125786163522014, 'inst_level_strict_acc_stderr,none': 'N/A', 'prompt_level_loose_acc,none': 0.115, 'prompt_level_loose_acc_stderr,none': 0.02261486592496228, 'inst_level_loose_acc,none': 0.2138364779874214, 'inst_level_loose_acc_stderr,none': 'N/A'}}

SFT gsm8k
{'gsm8k': {'name': 'gsm8k', 'alias': 'gsm8k', 'sample_len': 200, 'exact_match,strict-match': 0.0, 'exact_match_stderr,strict-match': 0.0, 'exact_match,flexible-extract': 0.475, 'exact_match_stderr,flexible-extract': 0.03539972744976421}}

SFT ifeval
{'ifeval': {'name': 'ife

In [35]:
import glob
import json
import os

for task in tasks:
    print("\n", task.upper())

    for name in models:
        files = glob.glob(f"results/{name}/{task}/**/samples_*.jsonl", recursive=True)

        if len(files) == 0:
            print(name, "none found")
            continue

        files.sort()
        file = files[-1]
        with open(file) as f:
            samples = []
            for line in f:
                samples.append(json.loads(line))

        sample = None

        for s in samples:
            if s["doc_id"] == 0:

                if task == "gsm8k" and s["filter"] != "flexible-extract":
                    continue

                sample = s
                break


        if sample is None:
            print(name, "question not found")
            continue

        if name == "Base":
            print("\nQuestion")
            question = sample["doc"].get("question", sample["doc"].get("prompt", ""))
            print(question)

        print("\n", name)
        print(str(sample["resps"][0][0])[:1500])


 GSM8K

Question
Janet’s ducks lay 16 eggs per day. She eats three for breakfast every morning and bakes muffins for her friends every day with four. She sells the remainder at the farmers' market daily for $2 per fresh duck egg. How much in dollars does she make every day at the farmers' market?

 Base
 16 eggs x $2 = $32


 SFT
She eats 3 eggs for breakfast and bakes 4 muffins for her friends so she uses 3 + 4 = 7 eggs per day. She has 16 - 7 = 9 eggs left to sell at the farmers' market. She sells each egg for $2 so she makes 9 * 2 = 18 dollars per day at the farmers' market. The final answer: 18.

 DPO
First find how many eggs Janet eats for breakfast: 3 eggs * $1 per egg = $3. Then subtract that number from the total number of eggs laid to find how many eggs are left for sale: 16 eggs - 3 eggs = 13 eggs. Then multiply the number of eggs left for sale by the price per egg to find Janet's daily earnings: 13 eggs * $2 per egg = $26. So Janet makes \boxed{26} dollars every day at the 